In [ ]:
from pathlib import Path
import openslide
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def select_level_tiles_from_coarse_thumbnail_contour(
    slide_path,
    target_level=4,
    tile_w=800,
    tile_h=1400,
    # --- detección gruesa del contorno en thumbnail ---
    thumb_sat_threshold=15,
    thumb_val_threshold=220,
    thumb_kernel_size=5,
    contour_band_thickness_thumb=10,
    # --- filtro de tiles en level ---
    specimen_frac_threshold=0.02,
    dist_threshold=18,
    sat_threshold=18,
    value_threshold=220,
    min_contour_overlap_pixels=5,
    show_removed=True,
):
    """
    Pipeline:
    1) Detecta contorno aproximado del espécimen en thumbnail con cv2
    2) Genera una banda de contorno en thumbnail
    3) Segmenta el espécimen en target_level para saber qué tiles contienen tejido
    4) Construye tiles en target_level
    5) Conserva solo tiles que:
       - no son 100% negros
       - contienen espécimen suficiente
       - intersectan la banda de contorno detectada en thumbnail
    6) Muestra:
       - máscara gruesa en thumbnail
       - contorno grueso en thumbnail
       - todas las cuadrículas
       - cuadrículas seleccionadas por contorno
       - tiles seleccionados uno por uno en level4
    """

    # =========================================================
    # 0. Abrir slide
    # =========================================================
    slide_path = Path(slide_path)
    slide = openslide.OpenSlide(str(slide_path))

    print("Level dimensions:", slide.level_dimensions)
    print("Level downsamples:", slide.level_downsamples)

    lvl_w, lvl_h = slide.level_dimensions[target_level]
    print(f"\nAnalizando level {target_level}: {lvl_w} x {lvl_h} px")

    # =========================================================
    # 1. Leer thumbnail
    # =========================================================
    thumb = slide.associated_images["thumbnail"].convert("RGB")
    thumb_np = np.array(thumb)
    thumb_h, thumb_w = thumb_np.shape[:2]

    # =========================================================
    # 2. Detectar contorno aproximado en thumbnail (cv2 HSV)
    # =========================================================
    hsv_thumb = cv2.cvtColor(thumb_np, cv2.COLOR_RGB2HSV)
    _, S_thumb, V_thumb = cv2.split(hsv_thumb)

    mask_tissue_thumb = ((S_thumb > thumb_sat_threshold) | (V_thumb < thumb_val_threshold)).astype(np.uint8) * 255

    kernel_thumb = np.ones((thumb_kernel_size, thumb_kernel_size), np.uint8)
    mask_tissue_thumb = cv2.morphologyEx(mask_tissue_thumb, cv2.MORPH_OPEN, kernel_thumb)
    mask_tissue_thumb = cv2.morphologyEx(mask_tissue_thumb, cv2.MORPH_CLOSE, kernel_thumb)

    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask_tissue_thumb, connectivity=8)
    if nlab <= 1:
        raise ValueError("No se ha detectado espécimen en la thumbnail.")

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask_thumb_main = np.uint8(labels == largest) * 255

    # rellenar huecos internos
    h_t, w_t = mask_thumb_main.shape
    flood = mask_thumb_main.copy()
    flood_mask = np.zeros((h_t + 2, w_t + 2), np.uint8)
    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    mask_thumb_filled = cv2.bitwise_or(mask_thumb_main, holes)

    contours_thumb, _ = cv2.findContours(mask_thumb_filled, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if len(contours_thumb) == 0:
        raise ValueError("No se ha encontrado contorno externo en thumbnail.")

    coarse_contour_thumb = max(contours_thumb, key=cv2.contourArea)

    contour_band_thumb = np.zeros_like(mask_thumb_filled)
    cv2.drawContours(
        contour_band_thumb,
        [coarse_contour_thumb],
        -1,
        255,
        thickness=contour_band_thickness_thumb
    )

    # =========================================================
    # 3. Mostrar máscara gruesa y contorno grueso en thumbnail
    # =========================================================
    plt.figure(figsize=(8, 14))
    plt.imshow(mask_thumb_filled, cmap="gray")
    plt.axis("off")
    plt.title("Máscara gruesa del espécimen en thumbnail")
    plt.show()

    overlay_thumb_contour = thumb_np.copy()
    cv2.drawContours(overlay_thumb_contour, [coarse_contour_thumb], -1, (255, 255, 0), 2)

    plt.figure(figsize=(8, 14))
    plt.imshow(overlay_thumb_contour)
    plt.axis("off")
    plt.title("Contorno grueso detectado en thumbnail")
    plt.show()

    # =========================================================
    # 4. Leer imagen completa del target_level y segmentar espécimen
    # =========================================================
    img_level = slide.read_region((0, 0), target_level, (lvl_w, lvl_h)).convert("RGB")
    img_level = np.array(img_level)

    img = img_level.astype(np.float32)

    border_pixels = np.concatenate([
        img[0, :, :],
        img[-1, :, :],
        img[:, 0, :],
        img[:, -1, :]
    ], axis=0)

    bg_color = np.median(border_pixels, axis=0)
    dist = np.linalg.norm(img - bg_color[None, None, :], axis=2)

    hsv_level = cv2.cvtColor(img_level, cv2.COLOR_RGB2HSV)
    _, S, V = cv2.split(hsv_level)

    mask = ((dist > dist_threshold) | (S > sat_threshold) | (V < value_threshold)).astype(np.uint8) * 255

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if nlab <= 1:
        raise ValueError("No se ha detectado espécimen en el target_level.")

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask_main = np.uint8(labels == largest) * 255

    # rellenar huecos internos
    h, w = mask_main.shape
    flood = mask_main.copy()
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    mask_filled = cv2.bitwise_or(mask_main, holes)

    # =========================================================
    # 5. Construir tiles en target_level
    # =========================================================
    tiles = []
    tile_id = 1

    scale_x = thumb_w / lvl_w
    scale_y = thumb_h / lvl_h

    row_id = 0
    for y0 in range(0, lvl_h, tile_h):
        col_id = 0
        for x0 in range(0, lvl_w, tile_w):
            w_tile = min(tile_w, lvl_w - x0)
            h_tile = min(tile_h, lvl_h - y0)

            tile_img = img_level[y0:y0 + h_tile, x0:x0 + w_tile]
            tile_mask = mask_filled[y0:y0 + h_tile, x0:x0 + w_tile]

            # 100% negro
            is_black = np.all(tile_img == 0)

            # espécimen dentro del tile
            specimen_frac = tile_mask.mean() / 255.0

            # proyección del tile sobre thumbnail
            x0_t = int(np.floor(x0 * scale_x))
            y0_t = int(np.floor(y0 * scale_y))
            x1_t = int(np.ceil((x0 + w_tile) * scale_x))
            y1_t = int(np.ceil((y0 + h_tile) * scale_y))

            x0_t = max(0, min(x0_t, thumb_w))
            x1_t = max(0, min(x1_t, thumb_w))
            y0_t = max(0, min(y0_t, thumb_h))
            y1_t = max(0, min(y1_t, thumb_h))

            if x1_t <= x0_t or y1_t <= y0_t:
                contour_overlap_pixels = 0
            else:
                contour_overlap_pixels = int(
                    np.count_nonzero(contour_band_thumb[y0_t:y1_t, x0_t:x1_t])
                )

            keep = (
                (not is_black)
                and (specimen_frac > specimen_frac_threshold)
                and (contour_overlap_pixels >= min_contour_overlap_pixels)
            )

            reason = []
            if is_black:
                reason.append("100% negra")
            if specimen_frac <= specimen_frac_threshold:
                reason.append(f"sin espécimen suficiente ({specimen_frac:.3f})")
            if contour_overlap_pixels < min_contour_overlap_pixels:
                reason.append(f"sin solape suficiente con contorno ({contour_overlap_pixels})")

            reason_text = "conservar" if keep else ", ".join(reason)

            tiles.append({
                "id": tile_id,
                "row": row_id,
                "col": col_id,
                "x0": x0,
                "y0": y0,
                "w": w_tile,
                "h": h_tile,
                "img": tile_img,
                "is_black": is_black,
                "specimen_frac": specimen_frac,
                "contour_overlap_pixels": contour_overlap_pixels,
                "keep": keep,
                "reason": reason_text,
            })

            tile_id += 1
            col_id += 1
        row_id += 1

    kept_tiles = [t for t in tiles if t["keep"]]
    removed_tiles = [t for t in tiles if not t["keep"]]

    # =========================================================
    # 6. Overview: todas las cuadrículas
    # =========================================================
    fig, ax = plt.subplots(figsize=(8, 14))
    ax.imshow(thumb_np)

    # dibujar contorno aproximado
    cv2_contour = coarse_contour_thumb[:, 0, :]
    ax.plot(cv2_contour[:, 0], cv2_contour[:, 1], color="yellow", linewidth=1.5)

    for t in tiles:
        x_t = t["x0"] * scale_x
        y_t = t["y0"] * scale_y
        w_t = t["w"] * scale_x
        h_t = t["h"] * scale_y

        rect = Rectangle((x_t, y_t), w_t, h_t, fill=False, edgecolor="cyan", linewidth=0.8)
        ax.add_patch(rect)

        ax.text(
            x_t + 2, y_t + 12, str(t["id"]),
            color="yellow", fontsize=7,
            bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
        )

    ax.set_title(f"Todas las cuadrículas - level {target_level} + contorno grueso")
    ax.axis("off")
    plt.show()

    # =========================================================
    # 7. Overview final: seleccionadas por el contorno
    # =========================================================
    fig, ax = plt.subplots(figsize=(8, 14))
    ax.imshow(thumb_np)

    # banda del contorno
    ax.contour(contour_band_thumb > 0, levels=[0.5], colors="yellow", linewidths=1)

    for t in tiles:
        x_t = t["x0"] * scale_x
        y_t = t["y0"] * scale_y
        w_t = t["w"] * scale_x
        h_t = t["h"] * scale_y

        if not t["keep"]:
            if not show_removed:
                continue
            color = "red"
            linestyle = "--"
        else:
            color = "lime"
            linestyle = "-"

        rect = Rectangle(
            (x_t, y_t), w_t, h_t,
            fill=False, edgecolor=color,
            linewidth=1.2, linestyle=linestyle
        )
        ax.add_patch(rect)

        ax.text(
            x_t + 2, y_t + 12, str(t["id"]),
            color=color,
            fontsize=7,
            bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
        )

    ax.set_title(
        f"Tiles seleccionados por contorno aproximado - level {target_level}\n"
        f"Verde = conservar | Rojo = eliminar"
    )
    ax.axis("off")
    plt.show()

    # =========================================================
    # 8. Texto
    # =========================================================
    print("\n==============================")
    print("TILES ELIMINADOS")
    print("==============================")
    if len(removed_tiles) == 0:
        print("Ninguno")
    else:
        for t in removed_tiles:
            print(
                f"Tile {t['id']:02d} | "
                f"row={t['row']}, col={t['col']} | "
                f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
                f"motivo: {t['reason']}"
            )

    print("\n==============================")
    print("TILES CONSERVADOS POR CONTORNO")
    print("==============================")
    if len(kept_tiles) == 0:
        print("Ninguno")
    else:
        for t in kept_tiles:
            print(
                f"Tile {t['id']:02d} | "
                f"row={t['row']}, col={t['col']} | "
                f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
                f"specimen_frac={t['specimen_frac']:.3f} | "
                f"contour_overlap_pixels={t['contour_overlap_pixels']}"
            )

    # =========================================================
    # 9. Mostrar tiles conservados uno por uno en target_level
    # =========================================================
    print("\n==============================")
    print("MOSTRANDO TILES CONSERVADOS")
    print("==============================")

    for t in kept_tiles:
        h_img, w_img = t["img"].shape[:2]
        aspect = h_img / w_img

        fig_w = 10
        fig_h = min(16, max(6, fig_w * aspect))

        plt.figure(figsize=(fig_w, fig_h))
        plt.imshow(t["img"])
        plt.axis("off")
        plt.title(
            f"Tile {t['id']} - level {target_level}\n"
            f"row={t['row']}, col={t['col']} | "
            f"x={t['x0']}, y={t['y0']}, w={t['w']}, h={t['h']} | "
            f"specimen_frac={t['specimen_frac']:.3f} | "
            f"contour_overlap_pixels={t['contour_overlap_pixels']}"
        )
        plt.show()

    return kept_tiles, removed_tiles, mask_thumb_filled, coarse_contour_thumb, contour_band_thumb, mask_filled

In [ ]:
slide_path = r"Datos\SB013\SB013\H&E_python\SB013.mrxs"

kept_tiles, removed_tiles, mask_thumb_filled, coarse_contour_thumb, contour_band_thumb, mask_filled = \
    select_level_tiles_from_coarse_thumbnail_contour(
        slide_path=slide_path,
        target_level=4,
        tile_w=800,
        tile_h=1400,
        specimen_frac_threshold=0.02,
        contour_band_thickness_thumb=10,
        min_contour_overlap_pixels=5
    )

In [ ]:
from pathlib import Path
import openslide
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd


def build_coarse_specimen_mask_on_level(
    img_level,
    dist_threshold=18,
    sat_threshold=18,
    value_threshold=220,
    kernel_size=5,
):
    """
    Máscara gruesa del espécimen en el level objetivo.
    """
    img = img_level.astype(np.float32)

    border_pixels = np.concatenate([
        img[0, :, :],
        img[-1, :, :],
        img[:, 0, :],
        img[:, -1, :]
    ], axis=0)

    bg_color = np.median(border_pixels, axis=0)
    dist = np.linalg.norm(img - bg_color[None, None, :], axis=2)

    hsv = cv2.cvtColor(img_level, cv2.COLOR_RGB2HSV)
    _, S, V = cv2.split(hsv)

    mask = ((dist > dist_threshold) | (S > sat_threshold) | (V < value_threshold)).astype(np.uint8) * 255

    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if nlab <= 1:
        raise ValueError("No se ha detectado espécimen en el level objetivo.")

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask_main = np.uint8(labels == largest) * 255

    # rellenar huecos internos
    h, w = mask_main.shape
    flood = mask_main.copy()
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    mask_filled = cv2.bitwise_or(mask_main, holes)

    return mask_filled


def refine_tile_tissue_from_white_background(
    tile_img,
    coarse_tile_mask,
    bg_sat_max=25,
    bg_val_min=210,
    bg_rgb_std_max=20,
    bg_open_kernel=3,
    bg_close_kernel=5,
    min_bg_component_area=50,
    tissue_open_kernel=3,
    tissue_close_kernel=5,
    min_tissue_component_area=100,
):
    """
    En una tile de borde:
    - detecta background blanco exterior
    - define tejido como coarse_mask - background exterior
    - devuelve:
      tissue_mask, outside_bg_mask, bg_candidate_mask
    """

    # ---------------------------------------------------------
    # 1. Candidatos a background blanco
    # ---------------------------------------------------------
    img_f = tile_img.astype(np.float32)
    hsv = cv2.cvtColor(tile_img, cv2.COLOR_RGB2HSV)
    _, S, V = cv2.split(hsv)

    rgb_std = np.std(img_f, axis=2)
    black_mask = np.all(tile_img < 10, axis=2)

    bg_candidate = (
        (S < bg_sat_max) &
        (V > bg_val_min) &
        (rgb_std < bg_rgb_std_max) &
        (~black_mask)
    ).astype(np.uint8) * 255

    k1 = np.ones((bg_open_kernel, bg_open_kernel), np.uint8)
    k2 = np.ones((bg_close_kernel, bg_close_kernel), np.uint8)

    bg_candidate = cv2.morphologyEx(bg_candidate, cv2.MORPH_OPEN, k1)
    bg_candidate = cv2.morphologyEx(bg_candidate, cv2.MORPH_CLOSE, k2)

    # ---------------------------------------------------------
    # 2. Quedarse solo con componentes blancas conectadas al borde
    #    => background exterior real
    # ---------------------------------------------------------
    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(bg_candidate, connectivity=8)

    outside_bg = np.zeros_like(bg_candidate)

    h, w = bg_candidate.shape
    for lab in range(1, nlab):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area < min_bg_component_area:
            continue

        comp = (labels == lab)
        ys, xs = np.where(comp)
        if len(ys) == 0:
            continue

        touches_image_border = (
            ys.min() == 0 or ys.max() == h - 1 or
            xs.min() == 0 or xs.max() == w - 1
        )

        if touches_image_border:
            outside_bg[comp] = 255

    # ---------------------------------------------------------
    # 3. Tejido refinado = máscara gruesa - background exterior
    # ---------------------------------------------------------
    tissue_mask = np.where(
        (coarse_tile_mask > 0) & (outside_bg == 0),
        255,
        0
    ).astype(np.uint8)

    # limpieza del tejido
    kt1 = np.ones((tissue_open_kernel, tissue_open_kernel), np.uint8)
    kt2 = np.ones((tissue_close_kernel, tissue_close_kernel), np.uint8)

    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_OPEN, kt1)
    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kt2)

    # quitar componentes muy pequeñas
    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(tissue_mask, connectivity=8)
    cleaned = np.zeros_like(tissue_mask)

    for lab in range(1, nlab):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= min_tissue_component_area:
            cleaned[labels == lab] = 255

    # fallback: si se ha quedado vacío, usar la coarse
    if np.count_nonzero(cleaned) == 0:
        cleaned = coarse_tile_mask.copy()

    return cleaned, outside_bg, bg_candidate


def build_precise_level4_contour_from_kept_tiles(
    slide_path,
    kept_tiles,
    target_level=4,
    coarse_global_mask=None,

    # --- coarse global mask, si no se pasa ---
    coarse_dist_threshold=18,
    coarse_sat_threshold=18,
    coarse_value_threshold=220,
    coarse_kernel_size=5,

    # --- detección de blanco exterior en tile ---
    bg_sat_max=25,
    bg_val_min=210,
    bg_rgb_std_max=20,
    bg_open_kernel=3,
    bg_close_kernel=5,
    min_bg_component_area=50,

    # --- limpieza tejido refinado ---
    tissue_open_kernel=3,
    tissue_close_kernel=5,
    min_tissue_component_area=100,

    # --- unión global final ---
    global_close_kernel=15,
    global_open_kernel=5,
    final_dilate_kernel=3,
    final_dilate_iters=0,

    # --- simplificación del contorno ---
    simplify_epsilon_ratio=0.001,

    # --- debug / guardado ---
    show_tile_debug=True,
    save_dir=None,
):
    """
    A partir de kept_tiles (seleccionadas por el contorno aproximado),
    reconstruye una máscara refinada en level4 usando la separación:
    espécimen vs background blanco exterior.

    Devuelve:
    - refined_global_mask
    - final_mask
    - final_contour
    - final_contour_simple
    - saved_files
    """

    slide_path = Path(slide_path)
    slide = openslide.OpenSlide(str(slide_path))

    lvl_w, lvl_h = slide.level_dimensions[target_level]
    ds = slide.level_downsamples[target_level]

    img_level = slide.read_region((0, 0), target_level, (lvl_w, lvl_h)).convert("RGB")
    img_level = np.array(img_level)

    # ---------------------------------------------------------
    # 1. Máscara gruesa global
    # ---------------------------------------------------------
    if coarse_global_mask is None:
        coarse_global_mask = build_coarse_specimen_mask_on_level(
            img_level,
            dist_threshold=coarse_dist_threshold,
            sat_threshold=coarse_sat_threshold,
            value_threshold=coarse_value_threshold,
            kernel_size=coarse_kernel_size,
        )

    # partimos de la máscara gruesa y la vamos refinando en las tiles seleccionadas
    refined_global_mask = coarse_global_mask.copy()

    debug_tiles = []

    # ---------------------------------------------------------
    # 2. Refinado tile a tile
    # ---------------------------------------------------------
    print("\n==============================")
    print("REFINANDO TILES EN LEVEL 4")
    print("==============================")

    for t in kept_tiles:
        x0, y0, w, h = t["x0"], t["y0"], t["w"], t["h"]

        tile_img = t["img"]
        coarse_tile_mask = coarse_global_mask[y0:y0+h, x0:x0+w]

        tissue_mask, outside_bg, bg_candidate = refine_tile_tissue_from_white_background(
            tile_img,
            coarse_tile_mask,
            bg_sat_max=bg_sat_max,
            bg_val_min=bg_val_min,
            bg_rgb_std_max=bg_rgb_std_max,
            bg_open_kernel=bg_open_kernel,
            bg_close_kernel=bg_close_kernel,
            min_bg_component_area=min_bg_component_area,
            tissue_open_kernel=tissue_open_kernel,
            tissue_close_kernel=tissue_close_kernel,
            min_tissue_component_area=min_tissue_component_area,
        )

        # sustituir esa tile en la máscara global
        refined_global_mask[y0:y0+h, x0:x0+w] = tissue_mask

        debug_tiles.append({
            "id": t["id"],
            "x0": x0,
            "y0": y0,
            "w": w,
            "h": h,
            "img": tile_img,
            "coarse_mask": coarse_tile_mask,
            "bg_candidate": bg_candidate,
            "outside_bg": outside_bg,
            "tissue_mask": tissue_mask,
        })

    # ---------------------------------------------------------
    # 3. Limpieza / unión global
    # ---------------------------------------------------------
    kc = np.ones((global_close_kernel, global_close_kernel), np.uint8)
    ko = np.ones((global_open_kernel, global_open_kernel), np.uint8)

    final_mask = cv2.morphologyEx(refined_global_mask, cv2.MORPH_CLOSE, kc)
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_OPEN, ko)

    if final_dilate_iters > 0:
        kd = np.ones((final_dilate_kernel, final_dilate_kernel), np.uint8)
        final_mask = cv2.dilate(final_mask, kd, iterations=final_dilate_iters)

    # quedarnos con el mayor componente
    nlab, labels, stats, _ = cv2.connectedComponentsWithStats(final_mask, connectivity=8)
    if nlab <= 1:
        raise ValueError("No se ha podido construir una máscara global final.")

    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    final_mask = np.uint8(labels == largest) * 255

    # rellenar huecos internos
    h, w = final_mask.shape
    flood = final_mask.copy()
    flood_mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, flood_mask, (0, 0), 255)
    holes = cv2.bitwise_not(flood)
    final_mask = cv2.bitwise_or(final_mask, holes)

    # ---------------------------------------------------------
    # 4. Contorno final
    # ---------------------------------------------------------
    contours, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if len(contours) == 0:
        raise ValueError("No se ha encontrado contorno final.")

    final_contour = max(contours, key=cv2.contourArea)

    eps = simplify_epsilon_ratio * cv2.arcLength(final_contour, True)
    final_contour_simple = cv2.approxPolyDP(final_contour, eps, True)

    # ---------------------------------------------------------
    # 5. Visualización local tile a tile
    # ---------------------------------------------------------
    if show_tile_debug:
        print("\n==============================")
        print("MOSTRANDO DEBUG TILE A TILE")
        print("==============================")

        for d in debug_tiles:
            h_img, w_img = d["img"].shape[:2]
            aspect = h_img / w_img
            fig_w = 14
            fig_h = min(12, max(5, fig_w * aspect / 2.3))

            fig, axes = plt.subplots(2, 3, figsize=(fig_w, fig_h))
            axes = axes.ravel()

            axes[0].imshow(d["img"])
            axes[0].axis("off")
            axes[0].set_title(f"Tile {d['id']} - RGB")

            axes[1].imshow(d["coarse_mask"], cmap="gray")
            axes[1].axis("off")
            axes[1].set_title("Coarse mask")

            axes[2].imshow(d["bg_candidate"], cmap="gray")
            axes[2].axis("off")
            axes[2].set_title("Blanco candidato")

            axes[3].imshow(d["outside_bg"], cmap="gray")
            axes[3].axis("off")
            axes[3].set_title("Background exterior")

            axes[4].imshow(d["tissue_mask"], cmap="gray")
            axes[4].axis("off")
            axes[4].set_title("Tejido refinado")

            overlay = d["img"].copy()
            contours_tile, _ = cv2.findContours(d["tissue_mask"], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            cv2.drawContours(overlay, contours_tile, -1, (255, 255, 0), 2)

            axes[5].imshow(overlay)
            axes[5].axis("off")
            axes[5].set_title("Contorno tile")

            plt.tight_layout()
            plt.show()

    # ---------------------------------------------------------
    # 6. Visualizaciones globales
    # ---------------------------------------------------------
    plt.figure(figsize=(8, 14))
    plt.imshow(coarse_global_mask, cmap="gray")
    plt.axis("off")
    plt.title("Máscara gruesa global en level 4")
    plt.show()

    plt.figure(figsize=(8, 14))
    plt.imshow(refined_global_mask, cmap="gray")
    plt.axis("off")
    plt.title("Máscara global tras refinar tiles")
    plt.show()

    plt.figure(figsize=(8, 14))
    plt.imshow(final_mask, cmap="gray")
    plt.axis("off")
    plt.title("Máscara global final")
    plt.show()

    overlay_level = img_level.copy()
    cv2.drawContours(overlay_level, [final_contour], -1, (255, 255, 0), 4)
    cv2.drawContours(overlay_level, [final_contour_simple], -1, (0, 255, 255), 2)

    plt.figure(figsize=(8, 14))
    plt.imshow(overlay_level)
    plt.axis("off")
    plt.title("Contorno final sobre H&E level 4\nAmarillo = denso | Cian = simplificado")
    plt.show()

    # thumbnail
    thumb = slide.associated_images["thumbnail"].convert("RGB")
    thumb_np = np.array(thumb)
    thumb_h, thumb_w = thumb_np.shape[:2]

    scale_x = thumb_w / lvl_w
    scale_y = thumb_h / lvl_h

    contour_thumb = np.column_stack([
        final_contour[:, 0, 0] * scale_x,
        final_contour[:, 0, 1] * scale_y
    ]).astype(np.int32)

    contour_thumb_simple = np.column_stack([
        final_contour_simple[:, 0, 0] * scale_x,
        final_contour_simple[:, 0, 1] * scale_y
    ]).astype(np.int32)

    overlay_thumb = thumb_np.copy()
    cv2.polylines(overlay_thumb, [contour_thumb.reshape(-1, 1, 2)], True, (255, 255, 0), 2)
    cv2.polylines(overlay_thumb, [contour_thumb_simple.reshape(-1, 1, 2)], True, (0, 255, 255), 1)

    plt.figure(figsize=(8, 14))
    plt.imshow(overlay_thumb)
    plt.axis("off")
    plt.title("Contorno final sobre thumbnail\nAmarillo = denso | Cian = simplificado")
    plt.show()

    # ---------------------------------------------------------
    # 7. Guardado opcional
    # ---------------------------------------------------------
    saved_files = {}

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        # máscaras / overlays
        coarse_path = save_dir / f"coarse_mask_level{target_level}.png"
        refined_path = save_dir / f"refined_global_mask_level{target_level}.png"
        final_path = save_dir / f"final_mask_level{target_level}.png"
        overlay_level_path = save_dir / f"overlay_contour_level{target_level}.png"
        overlay_thumb_path = save_dir / "overlay_contour_thumbnail.png"

        cv2.imwrite(str(coarse_path), coarse_global_mask)
        cv2.imwrite(str(refined_path), refined_global_mask)
        cv2.imwrite(str(final_path), final_mask)
        cv2.imwrite(str(overlay_level_path), cv2.cvtColor(overlay_level, cv2.COLOR_RGB2BGR))
        cv2.imwrite(str(overlay_thumb_path), cv2.cvtColor(overlay_thumb, cv2.COLOR_RGB2BGR))

        # contorno level4
        contour_level4 = final_contour[:, 0, :]
        contour_df = pd.DataFrame(contour_level4, columns=["x_level4", "y_level4"])
        contour_df["point_id"] = np.arange(1, len(contour_df) + 1)

        contour_level4_csv = save_dir / f"contour_level{target_level}.csv"
        contour_df.to_csv(contour_level4_csv, index=False)

        # contorno equivalente en level0
        contour_level0 = np.round(contour_level4 * ds).astype(int)
        contour0_df = pd.DataFrame(contour_level0, columns=["x_level0", "y_level0"])
        contour0_df["point_id"] = np.arange(1, len(contour0_df) + 1)

        contour_level0_csv = save_dir / f"contour_level0_from_level{target_level}.csv"
        contour0_df.to_csv(contour_level0_csv, index=False)

        saved_files = {
            "coarse_mask": coarse_path,
            "refined_global_mask": refined_path,
            "final_mask": final_path,
            "overlay_level": overlay_level_path,
            "overlay_thumb": overlay_thumb_path,
            "contour_level4_csv": contour_level4_csv,
            "contour_level0_csv": contour_level0_csv,
        }

        print("\n==============================")
        print("ARCHIVOS GUARDADOS")
        print("==============================")
        for k, v in saved_files.items():
            print(f"{k}: {v}")

    # ---------------------------------------------------------
    # 8. Resumen
    # ---------------------------------------------------------
    print("\n==============================")
    print("RESUMEN")
    print("==============================")
    print(f"Tiles refinadas: {len(kept_tiles)}")
    print(f"Puntos del contorno final (denso): {len(final_contour)}")
    print(f"Puntos del contorno final (simplificado): {len(final_contour_simple)}")

    return {
        "coarse_global_mask": coarse_global_mask,
        "refined_global_mask": refined_global_mask,
        "final_mask": final_mask,
        "final_contour": final_contour,
        "final_contour_simple": final_contour_simple,
        "saved_files": saved_files,
    }

In [ ]:
kept_tiles, removed_tiles, mask_thumb_filled, coarse_contour_thumb, contour_band_thumb, mask_filled = \
    select_level_tiles_from_coarse_thumbnail_contour(
        slide_path=slide_path,
        target_level=4,
        tile_w=800,
        tile_h=1400,
        specimen_frac_threshold=0.02,
        contour_band_thickness_thumb=10,
        min_contour_overlap_pixels=5
    )

In [ ]:
slide_path = r"Datos\SB013\SB013\H&E_python\SB013.mrxs"

result = build_precise_level4_contour_from_kept_tiles(
    slide_path=slide_path,
    kept_tiles=kept_tiles,
    target_level=4,
    coarse_global_mask=mask_filled,   # usa la máscara gruesa global que ya tienes

    # detección del fondo blanco exterior
    bg_sat_max=25,
    bg_val_min=210,
    bg_rgb_std_max=20,
    bg_open_kernel=3,
    bg_close_kernel=5,
    min_bg_component_area=50,

    # limpieza del tejido
    tissue_open_kernel=3,
    tissue_close_kernel=5,
    min_tissue_component_area=100,

    # unión global
    global_close_kernel=15,
    global_open_kernel=5,
    final_dilate_kernel=3,
    final_dilate_iters=0,

    simplify_epsilon_ratio=0.001,

    show_tile_debug=True,
    save_dir=r"Datos\SB013\SB013\H&E_python\resultado_contorno_level4"
)